# Mat & Comp – ML 9: Una red neuronal desde cero con numpy
## Muchas neuronas, en capas, que aprenden juntas

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 8

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar por qué las redes usan la función **sigmoide** en lugar del escalón.
2. Escribir una **capa** de neuronas como una multiplicación de matrices con numpy.
3. Programar la **propagación hacia adelante** de una red con una capa oculta.
4. **Entrenar** la red con el descenso de gradiente de ML 3, calculando las pendientes de forma numérica.
5. Ver a la red resolver el **XOR** y separar datos con fronteras curvas, y entender por qué hace falta un método más rápido.

## 🔍 ¿Puede la red aprender sola?

En el reto de ML 8 armaste el XOR con **tres** neuronas: una O, una NAND y una Y que las combina. Funcionó, pero **tú** elegiste todos los pesos.

✏️ *Si te dieran 3 neuronas con pesos al azar y solo la tabla del XOR, ¿cómo harías para que encontraran solas los pesos correctos? ¿Sirve la regla del perceptrón para las neuronas de la primera capa, si nadie sabe qué «deberían» responder?*

**Mi idea:**

La regla del perceptrón solo corrige a una neurona cuando sabemos su respuesta correcta. Las neuronas **ocultas** (las de en medio) no tienen respuesta correcta: solo ayudan. Hoy vamos a usar otra idea, que ya conoces de ML 3: **medir el error de toda la red y bajarlo con descenso de gradiente**, moviendo cada peso un poquito.

## 1. Del escalón a la sigmoide

El descenso de gradiente necesita **pendientes**: saber si el error baja al mover un peso un poquito. Con el escalón eso no funciona: si mueves un peso un poquito, la salida casi siempre sigue siendo 0 o 1, ¡no cambia nada! La pendiente es 0 y el algoritmo no sabe hacia dónde ir.

La solución es una versión **suave** del escalón, la función **sigmoide**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def escalon(z):
    return np.where(z > 0, 1, 0)

z = np.linspace(-8, 8, 400)
plt.plot(z, escalon(z), label="escalón (ML 8)", linewidth=2)
plt.plot(z, sigmoide(z), label="sigmoide", linewidth=2)
plt.axhline(0.5, color="gray", linestyle=":")
plt.xlabel("z"); plt.ylabel("salida")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Una versión suave del escalón")
plt.show()

for valor in [-5, -1, 0, 1, 5]:
    print(f"sigmoide({valor:2}) = {sigmoide(valor):.3f}")

La sigmoide:
- da números entre 0 y 1, que podemos leer como una **probabilidad** («estoy 88 % seguro de que es 1»);
- vale 0.5 en z = 0, que es donde el escalón salta;
- cambia **poco a poco**, así que un cambio pequeño en un peso produce un cambio pequeño en la salida. ¡Ahora sí hay pendiente!

## 2. Una capa de neuronas es una multiplicación de matrices

En ML 8, una neurona calculaba `x @ w + b`, con `w` un vector de pesos. Una **capa** tiene varias neuronas que reciben las **mismas** entradas, cada una con sus propios pesos. Si ponemos los pesos de cada neurona como una **columna**, obtenemos una matriz `W`:

- `W` tiene una **fila por entrada** y una **columna por neurona**;
- `b` tiene un sesgo por neurona;
- `X @ W + b` calcula todas las neuronas para todos los ejemplos **a la vez**.

Comprobémoslo con la primera capa del reto de ML 8 (las neuronas O y NAND), ahora con sigmoide y con los pesos multiplicados por 10 para que la sigmoide se parezca al escalón:

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])                   # XOR

W1 = 10 * np.array([[1, -1],                 # pesos de la entrada x1 hacia (O, NAND)
                    [1, -1]])                # pesos de la entrada x2 hacia (O, NAND)
b1 = 10 * np.array([-0.5, 1.5])              # sesgos de (O, NAND)

oculta = sigmoide(X @ W1 + b1)
print("Forma de X:", X.shape, "  Forma de W1:", W1.shape, "  Forma del resultado:", oculta.shape)
print("Capa oculta (columna 0 = O, columna 1 = NAND):")
print(oculta.round(3))

Cada fila es un ejemplo y cada columna una neurona. Ahora la **segunda capa**: una sola neurona (la Y) que recibe las dos salidas de la capa oculta.

In [ ]:
W2 = 10 * np.array([[1],
                    [1]])
b2 = 10 * np.array([-1.5])

salida = sigmoide(oculta @ W2 + b2)
print("Salida de la red:", salida.ravel().round(3))
print("XOR esperado:    ", y)

¡Una red de dos capas, calculada con dos multiplicaciones de matrices! Este cálculo, de las entradas a la salida, se llama **propagación hacia adelante**.

```
 entradas        capa oculta        salida
   x1  ──┬──▶  (neurona O)    ──┐
         ╳                       ├──▶ (neurona Y) ──▶ ŷ
   x2  ──┴──▶  (neurona NAND) ──┘
```

## 3. La red como una función

Juntemos todo en una función. Guardamos los pesos en una **lista** `[W1, b1, W2, b2]` para poder recorrerlos después.

In [ ]:
def adelante(X, params):
    W1, b1, W2, b2 = params
    oculta = sigmoide(X @ W1 + b1)            # capa oculta
    return sigmoide(oculta @ W2 + b2).ravel() # capa de salida; ravel() la deja como lista simple

def iniciar(n_entradas, n_ocultas, semilla):
    rng = np.random.default_rng(semilla)
    return [rng.normal(0, 1, (n_entradas, n_ocultas)),   # W1
            np.zeros(n_ocultas),                          # b1
            rng.normal(0, 1, (n_ocultas, 1)),             # W2
            np.zeros(1)]                                  # b2

params = iniciar(2, 3, semilla=0)
print("Una red 2-3-1 con pesos al azar responde al XOR así:")
print(adelante(X, params).round(3))

Con pesos al azar la red responde cualquier cosa. Necesitamos medir **qué tan mal** lo hace. Usamos el **error cuadrático medio** de ML 2:

In [ ]:
def error(X, y, params):
    return np.mean((adelante(X, params) - y) ** 2)

print("Error con pesos al azar:", round(error(X, y, params), 4))

Contemos las «perillas» que la red puede ajustar: `W1` tiene 2 × 3 = 6 pesos, `b1` tiene 3, `W2` tiene 3 y `b2` tiene 1. En total, **13 parámetros**. Aprender es encontrar los 13 números que hacen el error lo más pequeño posible.

In [ ]:
print("Número de parámetros:", sum(A.size for A in params))

## 4. Aprender: descenso de gradiente con pendientes numéricas

En ML 3 calculamos la pendiente de una función moviendo un poquito la variable:

$$\text{pendiente} \approx \frac{f(x + h) - f(x - h)}{2h}$$

(Es la versión «centrada», más precisa que la de ML 3.) Hacemos lo mismo con **cada uno** de los 13 parámetros: lo movemos un poquito hacia arriba, medimos el error; lo movemos hacia abajo, medimos el error; y la diferencia nos da la pendiente para ese parámetro.

`np.ndindex(A.shape)` recorre todas las posiciones de una matriz: (0, 0), (0, 1), (0, 2), (1, 0)…

In [ ]:
def gradiente_numerico(X, y, params, h=1e-5):
    gradientes = []
    for A in params:                          # W1, b1, W2, b2
        G = np.zeros_like(A)                  # una pendiente por cada número de A
        for pos in np.ndindex(A.shape):
            original = A[pos]
            A[pos] = original + h
            error_mas = error(X, y, params)
            A[pos] = original - h
            error_menos = error(X, y, params)
            A[pos] = original                 # dejamos el parámetro como estaba
            G[pos] = (error_mas - error_menos) / (2 * h)
        gradientes.append(G)
    return gradientes

g = gradiente_numerico(X, y, params)
print("Pendientes del error respecto a W1:")
print(g[0].round(4))

Ahora el descenso de gradiente de ML 3: en cada paso, cada parámetro se mueve **en contra** de su pendiente.

$$\text{parámetro} \leftarrow \text{parámetro} - \alpha \cdot \text{pendiente}$$

In [ ]:
def entrenar(X, y, n_ocultas=3, alfa=2.0, pasos=3000, semilla=0):
    params = iniciar(X.shape[1], n_ocultas, semilla)
    historia = []
    for paso in range(pasos):
        grads = gradiente_numerico(X, y, params)
        for A, G in zip(params, grads):
            A -= alfa * G                     # descenso de gradiente
        historia.append(error(X, y, params))
    return params, historia

params, historia = entrenar(X, y)

print("Error final:", round(historia[-1], 5))
print("Salida de la red:", adelante(X, params).round(3))
print("XOR esperado:    ", y)

plt.plot(historia)
plt.xlabel("paso"); plt.ylabel("error cuadrático medio")
plt.title("Curva de aprendizaje de la red en el XOR")
plt.grid(alpha=0.3)
plt.show()

🎉 **¡La red aprendió el XOR sola!** Nadie le dijo qué debía hacer cada neurona oculta: solo le dimos la tabla y el error. Esto es lo que el perceptrón de ML 8 no podía hacer.

Fíjate en la curva: al principio el error casi no baja (una «meseta») y de pronto cae. Es muy común en las redes neuronales.

## 5. ¿Qué ve la red?

Dibujemos qué responde la red en **todo** el plano, no solo en los 4 puntos. A diferencia del perceptrón, la frontera ya no es una sola recta:

In [ ]:
def mapa(eje, X, y, params, lim=(-0.5, 1.5), titulo=""):
    a, c = np.meshgrid(np.linspace(*lim, 200), np.linspace(*lim, 200))
    rejilla = np.column_stack([a.ravel(), c.ravel()])
    P = adelante(rejilla, params).reshape(a.shape)
    im = eje.contourf(a, c, P, levels=np.linspace(0, 1, 11), cmap="RdYlGn", alpha=0.8)
    eje.contour(a, c, P, levels=[0.5], colors="black")
    eje.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, "darkgreen", "darkred"), s=60, edgecolors="white")
    eje.set_title(titulo)
    eje.set_aspect("equal")
    return im

fig, eje = plt.subplots(figsize=(5.5, 5))
im = mapa(eje, X, y, params, titulo="La red 2-3-1 entrenada (línea negra: 0.5)")
plt.colorbar(im, label="probabilidad de 1")
plt.show()

Los colores muestran la **probabilidad** que da la red: verde cerca de 1, rojo cerca de 0. La franja verde separa las dos esquinas rojas: algo imposible con una sola recta.

¿Y qué hace cada neurona oculta? Cada una, por separado, es como un perceptrón: dibuja **una recta suave**. La neurona de salida las **combina**.

In [ ]:
W1, b1, W2, b2 = params
a, c = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
fig, ejes = plt.subplots(1, W1.shape[1], figsize=(5 * W1.shape[1], 4.5))
for j, eje in enumerate(ejes):
    act = sigmoide(a * W1[0, j] + c * W1[1, j] + b1[j])
    eje.contourf(a, c, act, levels=np.linspace(0, 1, 11), cmap="Blues")
    eje.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, "darkgreen", "darkred"), s=60, edgecolors="white")
    eje.set_title(f"neurona oculta {j}  (peso hacia la salida: {W2[j, 0]:+.1f})")
    eje.set_aspect("equal")
plt.show()

✏️ *¿Alguna neurona oculta se parece a la O o a la NAND de ML 8? La red pudo haber encontrado otra combinación: hay muchas formas de resolver el XOR.*

## 6. Un problema más difícil: dos medias lunas

En ML 6 vimos que k-means no podía separar dos medias lunas. Veamos qué hace nuestra red, ahora con 8 neuronas ocultas y 200 puntos.

In [ ]:
from sklearn.datasets import make_moons

Xl, yl = make_moons(200, noise=0.15, random_state=0)

params_l, historia_l = entrenar(Xl, yl, n_ocultas=8, alfa=1.0, pasos=1500, semilla=0)
exactitud = np.mean((adelante(Xl, params_l) > 0.5) == yl)
print(f"Error final: {historia_l[-1]:.4f}   ·   Exactitud: {exactitud:.0%}")
print("Parámetros:", sum(A.size for A in params_l))

fig, eje = plt.subplots(figsize=(7, 5))
mapa(eje, Xl, yl, params_l, lim=(-1.5, 2.5), titulo="Red 2-8-1 con dos medias lunas")
plt.show()

La red dibuja una frontera **curva** que se mete entre las dos lunas (todavía algo tosca: con más pasos se ajusta mejor). Con suficientes neuronas ocultas, una red puede aproximar fronteras de casi cualquier forma.

✏️ *Esta celda tarda unos segundos. ¿Por qué crees que es tan lenta, si solo son 200 puntos?*

## 7. ⚠️ El problema: es muy lento

En cada paso, el gradiente numérico calcula el error **dos veces por cada parámetro**. Contemos:

In [ ]:
import time

for n_ocultas in [3, 8, 32, 128]:
    p = iniciar(2, n_ocultas, 0)
    n_params = sum(A.size for A in p)
    inicio = time.time()
    gradiente_numerico(Xl, yl, p)
    t = time.time() - inicio
    print(f"{n_ocultas:4} ocultas: {n_params:4} parámetros → {2 * n_params:4} cálculos del error por paso, {t * 1000:6.1f} ms por paso")

El tiempo crece con el número de parámetros. Una red que reconoce dígitos (ML 11) tiene unos **100 000 parámetros**, y los modelos de lenguaje modernos tienen **miles de millones**. Con el gradiente numérico, ¡un solo paso tomaría horas o años!

En 1986, David Rumelhart, Geoffrey Hinton y Ronald Williams popularizaron un método que calcula **todas** las pendientes a la vez, con el costo de apenas un par de propagaciones: la **retropropagación** (*backpropagation*). Usa la regla de la cadena para «devolver» el error desde la salida hacia atrás, capa por capa. Es el tema de ML 10, y es lo que hace posible el aprendizaje profundo de hoy.

## 8. Con scikit-learn

scikit-learn tiene redes neuronales listas en `MLPClassifier` («perceptrón multicapa»). Por dentro usa retropropagación, así que es mucho más rápido:

In [ ]:
from sklearn.neural_network import MLPClassifier

red = MLPClassifier(hidden_layer_sizes=(8,), activation="logistic", solver="lbfgs",
                    max_iter=2000, random_state=0)
red.fit(Xl, yl)
print(f"Exactitud: {red.score(Xl, yl):.0%}")
print("Formas de los pesos:", [W.shape for W in red.coefs_])

- `hidden_layer_sizes=(8,)`: una capa oculta con 8 neuronas. Con `(8, 8)` tendría dos capas ocultas.
- `activation="logistic"`: la sigmoide (su otro nombre es función logística).
- `coefs_` guarda las matrices `W1` y `W2`: las mismas formas (2, 8) y (8, 1) que las nuestras.

Llega al 100 % en un instante, mientras que la nuestra se quedó en 92 % después de 1500 pasos lentos: calcula las pendientes con retropropagación y usa un método de optimización más astuto que el descenso de gradiente simple (`lbfgs`).

## 9. Errores comunes

**Error 1: formas que no encajan.** En `X @ W`, el número de **columnas** de `X` debe ser igual al número de **filas** de `W`. Si no, numpy no puede multiplicar:

In [ ]:
W_mal = np.ones((3, 4))           # pensado para 3 entradas, pero X tiene 2 columnas
X @ W_mal

Truco: escribe siempre las formas en un comentario, como `# (4, 2) @ (2, 3) → (4, 3)`.

**Error 2: empezar con todos los pesos en cero.** Todas las neuronas ocultas serían idénticas, tendrían las mismas pendientes… ¡y seguirían idénticas para siempre! Por eso se empieza con pesos **al azar** (lo verás en «¿Qué pasa si…?»).

**Error 3: una tasa de aprendizaje inadecuada.** Como en ML 3: muy pequeña es lenta; muy grande hace que el error salte.

**Error 4: rendirse en la meseta.** Si el error no baja al principio, a veces solo hay que esperar más pasos (o probar otra semilla).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …calculas `sigmoide(100)` y `sigmoide(-100)`? ¿Y `sigmoide(np.array([0, 0.1]))`? ¿Cuánto cambia la salida?
2. …cambias `iniciar` para que todos los pesos empiecen en **cero** y entrenas el XOR? Mira la salida y los pesos de `W1`.
3. …entrenas el XOR con `n_ocultas=1`? ¿Puede una sola neurona oculta resolverlo?
4. …usas `alfa=0.1` o `alfa=20` en el XOR? Compara las curvas de aprendizaje.
5. …usas el **escalón** en lugar de la sigmoide dentro de `adelante` y calculas `gradiente_numerico`? ¿Qué pendientes obtienes?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). La sigmoide a mano.** Calcula con calculadora $\sigma(0)$, $\sigma(2)$ y $\sigma(-2)$. ¿Qué relación hay entre $\sigma(2)$ y $\sigma(-2)$? Compruébalo con numpy.

**Ejercicio 2 (fácil). Una red a mano.** Una red tiene `W1 = [[1, -1], [1, -1]]`, `b1 = [0, 0]`, `W2 = [[2], [2]]` y `b2 = [-2]`. Calcula a mano (con la calculadora para la sigmoide) su salida para la entrada `[1, 0]`. Luego compruébalo con `adelante`.

**Ejercicio 3 (medio). Otras compuertas.** Entrena la red 2-3-1 para que aprenda la compuerta **Y** y la compuerta **O**. ¿Aprende más rápido que el XOR? Compara las curvas de aprendizaje en una misma gráfica.

**Ejercicio 4 (medio). ¿Cuántas neuronas?** Con las medias lunas, entrena redes con 1, 2, 4 y 8 neuronas ocultas (1500 pasos, `alfa=1.0`) y dibuja los cuatro mapas lado a lado. ¿Con cuántas neuronas la frontera deja de ser casi una recta?

**Ejercicio 5 (medio). Círculos.** `make_circles(200, noise=0.1, factor=0.4, random_state=0)` (de `sklearn.datasets`) crea un círculo dentro de otro. Entrena el perceptrón de ML 8 (con `Perceptron` de scikit-learn) y nuestra red con 8 neuronas ocultas. Compara la exactitud y dibuja el mapa de la red.

## 🏆 Reto opcional: una red que dibuja una curva

Las redes también pueden predecir **números** (regresión). Quita la sigmoide de la capa de salida para que la red pueda dar cualquier número:

```python
def adelante_reg(X, params):
    W1, b1, W2, b2 = params
    return (sigmoide(X @ W1 + b1) @ W2 + b2).ravel()
```

1. Crea 40 puntos `x = np.linspace(-3, 3, 40).reshape(-1, 1)` con `y = np.sin(x).ravel()`.
2. Adapta `error`, `gradiente_numerico` y `entrenar` para usar `adelante_reg`, y entrena una red 1-10-1.
3. Dibuja los puntos y la curva de la red. ¿Cuántas neuronas ocultas necesitas para que se parezca al seno? ¿Qué pasa fuera del intervalo [−3, 3]?

## 🎬 Cierre

Hoy aprendiste que:
- La **sigmoide** es un escalón suave: permite calcular pendientes y leer la salida como una probabilidad.
- Una **capa** de neuronas es `sigmoide(X @ W + b)`: una multiplicación de matrices.
- La **propagación hacia adelante** encadena capas desde la entrada hasta la salida.
- Una red con una **capa oculta** aprende sola el XOR y fronteras curvas, usando descenso de gradiente sobre **todos** sus parámetros.
- Calcular las pendientes moviendo cada parámetro es **muy lento** para redes grandes.

**Para pensar:** si calcular la pendiente de cada parámetro por separado es tan lento, ¿habrá una forma de calcularlas **todas a la vez**? La clave es la **regla de la cadena** de las derivadas: si sabemos cómo cambia el error con la salida, y cómo cambia la salida con la capa oculta, podemos ir «hacia atrás». En la próxima lección: la **retropropagación**. ⏪